In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib.utils import ImageReader
import os

# === Chemins
plots_dir = '/home/amenacer/Stage/Data/Segmentation/resultas/plots'
pdf_path = '/home/amenacer/Stage/Data/Segmentation/resultas/rapport_segmentation.pdf'

# === Création du PDF
c = canvas.Canvas(pdf_path, pagesize=A4)
width, height = A4

# === Titre
c.setFont("Helvetica-Bold", 16)
c.drawCentredString(width / 2, height - 50, "Rapport de segmentation nnUNet")
c.setFont("Helvetica", 10)
c.drawString(50, height - 80, "Généré automatiquement par script Python")

# === Résumé
y = height - 120
c.setFont("Helvetica-Bold", 12)
c.drawString(50, y, "Statistiques générales :")
c.setFont("Helvetica", 10)
y -= 20

# Lecture des données depuis CSV
import pandas as pd
df = pd.read_csv('/home/amenacer/Stage/Data/Segmentation/resultas/recap_global.csv')
total = len(df)
groupes = df['Groupe'].nunique()
traitées = df[df['Résultat généré'] == True].shape[0]
sans_masque = df[df['Masque trouvé'] == False].shape[0]

stats = [
    f"Total d'images : {total}",
    f"Nombre de groupes : {groupes}",
    f"Images traitées : {traitées}",
    f"Images sans masque : {sans_masque}"
]

for line in stats:
    c.drawString(70, y, f"• {line}")
    y -= 15

# === Ajout des images
def insert_image(path, title, y_start):
    if os.path.exists(path):
        c.setFont("Helvetica-Bold", 12)
        c.drawString(50, y_start, title)
        y_start -= 10
        img = ImageReader(path)
        c.drawImage(img, 50, y_start - 200, width=500, height=200, preserveAspectRatio=True)
        return y_start - 230
    return y_start

y = insert_image(os.path.join(plots_dir, 'images_traitees_par_groupe.png'),
                 "1. Images traitées par groupe", y - 20)

y = insert_image(os.path.join(plots_dir, 'masques_manquants_par_groupe.png'),
                 "2. Masques manquants par groupe", y - 10)

y = insert_image(os.path.join(plots_dir, 'fichiers_totaux_par_groupe.png'),
                 "3. Nombre total de fichiers par groupe", y - 10)

# === Finaliser
c.showPage()
c.save()

print(f"📄 Rapport PDF généré : {pdf_path}")
